In [1]:
import os

In [2]:
%pwd

'/Users/Alex/DS projects/mySummerizer/research'

In [3]:
os.chdir("../")

In [4]:
%pwd


'/Users/Alex/DS projects/mySummerizer'

In [36]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str

In [37]:
from mySummarizer.constants import *
from mySummarizer.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(str(config_filepath))
        self.params = read_yaml(str(params_filepath))

        create_directories([self.config.artifacts_root])


    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        return DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            tokenizer_name=str(config.tokenizer_name),
        )

In [59]:
import os
from mySummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

In [60]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)


    
    def convert_examples_to_features(self,example_batch):
        input_encodings = self.tokenizer(example_batch['dialogue'] , max_length = 1024, truncation = True )
        
        with self.tokenizer.as_target_tokenizer():
            target_encodings = self.tokenizer(example_batch['summary'], max_length = 128, truncation = True )
            
        return {
            'input_ids' : input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }
    

    def convert(self):
        if not os.path.exists(self.config.data_path):
            raise FileNotFoundError(f"data_path not found: {self.config.data_path}")
        dataset_samsum = load_from_disk(str(self.config.data_path))
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched=True)
        dataset_samsum_pt.save_to_disk(str(self.config.root_dir / "samsum_dataset"))

In [62]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-03-20 20:48:02,622]: INFO: common: Successfully read YAML file: config/config.yaml
[2026-03-20 20:48:02,625]: INFO: common: Successfully read YAML file: params.yaml
[2026-03-20 20:48:02,626]: INFO: common: Created directory at: artifacts
[2026-03-20 20:48:02,627]: INFO: common: Created directory at: artifacts/data_transformation


Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 51160.73 examples/s]
